### 0. Problem restatement

You have:
$Number$ samples (rows)
$People = P$ participants (columns)
Each sample is a 红包分配结果: $$ x=(x_1,x_2,...,x_p) $$
with hard constraints:
$$
x_i \ge 0, \sum_{i=1}^{p} x_i=Sum
$$
#### Your goal:

Learn a probabilistic model that approximates the true distribution of allocations,
and generate new samples that:

are non-negative

sum exactly to Sum

preserve the statistical structure of the original data

### 1.Imports and configuration

In [53]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from tqdm import tqdm

# -----------------------
# Configuration
# -----------------------
CSV_PATH = "envelopes.csv"

Sum = float(input("输入每个红包的总金额"))       # total amount (given)
People = None      # inferred from data

BATCH_SIZE = 128
EPOCHS = 300
LR = 1e-3
T = 5000           # diffusion steps

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


### 2、Normalization and Log-ratio transformation

We first convert amounts into proportions:
$$
p_i=\frac{x_i}{Sum}
$$
Now each sample lies on the probability simplex.

To apply diffusion, we map the simplex to Euclidean space using the **Additive Log-Ratio (ALR) transform**.
Choose the last person as reference (arbitrary but fixed):$$y_i=log(\frac{p_i}{p_P}) (i=1,...,P-1)$$
This gives:$$y_i \in \mathbf{R}^{P-1}$$ 

In [54]:
# -----------------------
# Load data
# -----------------------
df = pd.read_csv(CSV_PATH, header=None)
df = df.iloc[1:, 1:]
x = df.values.astype(np.float32)

People = x.shape[1]
print("人数：",People)

# Convert to proportions
p = x / Sum

# -----------------------
# Centered Log-Ratio (CLR) transform
# -----------------------
# y_i = log(p_i / geometric_mean(p))
# Equivalent to: log(p_i) - mean(log(p))
eps = 1e-8
log_p = np.log(p + eps)
# Subtract the mean of logs along the last dimension (rows)
y = log_p - np.mean(log_p, axis=1, keepdims=True)

y = torch.tensor(y, dtype=torch.float32)
# Note: y shape is now (N_samples, People), not (N_samples, People-1)


人数： 8


### 3.Time Embedding

In [55]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t):
        t = t.float().unsqueeze(-1) / T
        return self.net(t)


### 4.MLP denoiser

In [56]:
class DenoiseMLP(nn.Module):
    def __init__(self, data_dim, hidden_dim=256):
        super().__init__()
        self.time_emb = TimeEmbedding(hidden_dim)

        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, data_dim)
        )

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        x = torch.cat([x, t_emb], dim=1)
        return self.net(x)


### 5. Diffusion process (noise prediction formulation)

In [57]:
class Diffusion:
    def __init__(self, model, T):
        self.model = model
        self.T = T

        self.betas = torch.linspace(1e-4, 0.02, T).to(DEVICE)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)

    # Helper for Heavy-tailed noise (Laplace)
    # Variance of Laplace(0, b) is 2b^2. To match unit variance of standard Gaussian (var=1),
    # we need 2b^2 = 1 => b = 1/sqrt(2) approx 0.707
    def get_noise(self, shape):
        # Using Laplace distribution for heavy tails
        b = 1.0 / np.sqrt(2) 
        return torch.distributions.Laplace(0.0, b).sample(shape).to(DEVICE)

    def q_sample(self, y0, t, noise):
        a_bar = self.alpha_bar[t].unsqueeze(1)
        return torch.sqrt(a_bar) * y0 + torch.sqrt(1 - a_bar) * noise

    def loss(self, y0):
        bsz = y0.size(0)
        t = torch.randint(0, self.T, (bsz,), device=DEVICE)
        
        # MODIFIED: Use heavy-tailed noise
        noise = self.get_noise(y0.shape)

        y_t = self.q_sample(y0, t, noise)
        noise_pred = self.model(y_t, t)

        # MODIFIED: Use L1 Loss (or Huber Loss) for robustness
        return nn.L1Loss()(noise_pred, noise)
        # return nn.HuberLoss()(noise_pred, noise) # Alternative

    @torch.no_grad()
    def sample(self, n_samples, data_dim):
        # Start from pure noise
        y = self.get_noise((n_samples, data_dim))

        for t in tqdm(reversed(range(self.T)), desc="Sampling"):
            t_batch = torch.full((n_samples,), t, device=DEVICE)

            beta = self.betas[t]
            alpha = self.alphas[t]
            a_bar = self.alpha_bar[t]

            eps = self.model(y, t_batch)
            mean = (1 / torch.sqrt(alpha)) * (
                y - beta / torch.sqrt(1 - a_bar) * eps
            )

            if t > 0:
                # Add noise at each step (Langevin dynamics style)
                # Ensure consistency with the forward process noise
                noise = self.get_noise(y.shape)
                y = mean + torch.sqrt(beta) * noise
            else:
                y = mean

        return y

### 6.Training

In [58]:
dataset = torch.utils.data.TensorDataset(y)
loader = torch.utils.data.DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True
)

model = DenoiseMLP(data_dim=People).to(DEVICE)

diffusion = Diffusion(model, T)
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    total_loss = 0.0
    for (y0,) in loader:
        y0 = y0.to(DEVICE)

        loss = diffusion.loss(y0)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:03d} | Loss: {total_loss / len(loader):.6f}")


Epoch 001 | Loss: 0.382770
Epoch 002 | Loss: 0.171441
Epoch 003 | Loss: 0.158489
Epoch 004 | Loss: 0.156943
Epoch 005 | Loss: 0.151157
Epoch 006 | Loss: 0.147501
Epoch 007 | Loss: 0.148782
Epoch 008 | Loss: 0.146742
Epoch 009 | Loss: 0.144025
Epoch 010 | Loss: 0.140924
Epoch 011 | Loss: 0.138144
Epoch 012 | Loss: 0.141916
Epoch 013 | Loss: 0.141627
Epoch 014 | Loss: 0.140934
Epoch 015 | Loss: 0.140318
Epoch 016 | Loss: 0.135469
Epoch 017 | Loss: 0.135691
Epoch 018 | Loss: 0.137752
Epoch 019 | Loss: 0.134548
Epoch 020 | Loss: 0.137243
Epoch 021 | Loss: 0.136897
Epoch 022 | Loss: 0.133186
Epoch 023 | Loss: 0.128062
Epoch 024 | Loss: 0.137160
Epoch 025 | Loss: 0.135725
Epoch 026 | Loss: 0.135347
Epoch 027 | Loss: 0.130037
Epoch 028 | Loss: 0.131798
Epoch 029 | Loss: 0.130919
Epoch 030 | Loss: 0.129575
Epoch 031 | Loss: 0.132672
Epoch 032 | Loss: 0.132642
Epoch 033 | Loss: 0.130194
Epoch 034 | Loss: 0.128924
Epoch 035 | Loss: 0.132123
Epoch 036 | Loss: 0.126835
Epoch 037 | Loss: 0.126094
E

### 7.Sampling and inverse transform (guaranteed sum)

In [62]:
# -----------------------
# Generate new samples
# -----------------------
n_samples = 10000

# MODIFIED: Sampling dimension is now 'People'
y_gen = diffusion.sample(n_samples, People)

# MODIFIED: Inverse CLR transform (Softmax)
# Unlike ALR, we don't need to append a column of ones first.
# We just exponentiate and normalize.
y_gen = y_gen.cpu().numpy()
exp_y = np.exp(y_gen)
p_all = exp_y / exp_y.sum(axis=1, keepdims=True)


# Recover continuous amounts FIRST
x_gen_continuous = Sum * p_all

def round_to_cents_preserve_sum(x, decimals=2):
    """
    x: numpy array of shape (People,)
    returns: rounded array with fixed decimals and preserved sum
    """
    scale = 10 ** decimals
    c = x * scale

    floor_c = np.floor(c)
    remainder = c - floor_c

    deficit = int(round(scale * x.sum() - floor_c.sum()))

    # indices of largest remainders
    idx = np.argsort(remainder)[::-1][:deficit]
    floor_c[idx] += 1

    return floor_c / scale


# Recover amounts
x_gen = np.vstack([
    round_to_cents_preserve_sum(row, decimals=2)
    for row in x_gen_continuous
])


# Save
pd.DataFrame(x_gen).to_csv(
    "generated_samples.csv",
    index=False,
    header=False
)

print("Generated samples saved. Total sum per row:",
      np.unique(x_gen.sum(axis=1)))


Sampling: 5000it [01:24, 59.13it/s]


Generated samples saved. Total sum per row: [39.999996 40.       40.000004]
